In [ ]:
# Databricks notebook source

# =========================
# SETUP
# =========================
catalog = "cabpluse360_team2"
silver_schema = "cabpulse360_silver_team2"
gold_schema = "cabpulse360_gold_team2"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {gold_schema}")

print(f"Using catalog: {catalog}")
print(f"Using schema: {gold_schema}")

# =========================
# IMPORTS
# =========================
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ============================================================
# DIM_DATE
# ============================================================
df_dates = spark.table(f"{catalog}.{silver_schema}.trips") \
    .select("TRIP_DATE").distinct() \
    .withColumnRenamed("TRIP_DATE", "DATE_ACTUAL")

df_dim_date = df_dates \
    .withColumn("DATE_KEY", date_format("DATE_ACTUAL", "yyyyMMdd").cast("int")) \
    .withColumn("YEAR", year("DATE_ACTUAL")) \
    .withColumn("MONTH", month("DATE_ACTUAL")) \
    .withColumn("DAY", dayofmonth("DATE_ACTUAL"))

df_dim_date.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_date")

print("dim_date created")

# ============================================================
# DIM_VENDOR (SCD2 FIXED)
# ============================================================
df_vendors = spark.table(f"{catalog}.{silver_schema}.vendors") \
    .select("VENDOR_ID", "VENDOR_NAME")

df_changes = spark.table(f"{catalog}.{silver_schema}.vendor_name_changes")

df_changed = df_changes.join(df_vendors, "VENDOR_ID")

# OLD RECORD
df_old = df_changed.select(
    col("VENDOR_ID"),
    col("OLD_NAME").alias("VENDOR_NAME"),
    lit("1900-01-01").cast("date").alias("EFFECTIVE_DATE"),
    date_sub("CHANGE_DATE", 1).alias("END_DATE"),
    lit(0).alias("IS_CURRENT")
)

# NEW RECORD
df_new = df_changed.select(
    col("VENDOR_ID"),
    col("NEW_NAME").alias("VENDOR_NAME"),
    col("CHANGE_DATE").alias("EFFECTIVE_DATE"),
    lit("9999-12-31").cast("date").alias("END_DATE"),
    lit(1).alias("IS_CURRENT")
)

# UNCHANGED RECORD
df_unchanged = df_vendors.join(df_changes, "VENDOR_ID", "left_anti") \
    .select(
        col("VENDOR_ID"),
        col("VENDOR_NAME"),
        lit("1900-01-01").cast("date").alias("EFFECTIVE_DATE"),
        lit("9999-12-31").cast("date").alias("END_DATE"),
        lit(1).alias("IS_CURRENT")
    )

# UNION SAFE
df_dim_vendor = df_old.unionByName(df_new).unionByName(df_unchanged)

# SURROGATE KEY
window_sk = Window.partitionBy("VENDOR_ID").orderBy("EFFECTIVE_DATE")

df_dim_vendor = df_dim_vendor.withColumn(
    "VENDOR_SK", row_number().over(window_sk)
)

df_dim_vendor.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_vendor")

print("dim_vendor created")

# ============================================================
# DIM_TAXI_ZONE
# ============================================================
df_zone = spark.table(f"{catalog}.{silver_schema}.taxi_zones")

window_z = Window.orderBy("LOCATION_ID")

df_dim_zone = df_zone \
    .withColumn("ZONE_SK", row_number().over(window_z)) \
    .select("ZONE_SK", "LOCATION_ID", "BOROUGH", "ZONE")

df_dim_zone.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_taxi_zone")

print("dim_taxi_zone created")

# ============================================================
# DIM_RATE_CODE
# ============================================================
df_rate = spark.table(f"{catalog}.{silver_schema}.rate_codes")

window_r = Window.orderBy("RATE_CODE_ID")

df_dim_rate = df_rate \
    .withColumn("RATE_CODE_SK", row_number().over(window_r)) \
    .select("RATE_CODE_SK", "RATE_CODE_ID", "RATE_CODE_NAME")

df_dim_rate.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_rate_code")

print("dim_rate_code created")

# ============================================================
# DIM_PAYMENT_TYPE
# ============================================================
df_pay = spark.table(f"{catalog}.{silver_schema}.payment_types")

window_p = Window.orderBy("PAYMENT_TYPE_ID")

df_dim_pay = df_pay \
    .withColumn("PAYMENT_TYPE_SK", row_number().over(window_p)) \
    .select("PAYMENT_TYPE_SK", "PAYMENT_TYPE_ID", "PAYMENT_TYPE_NAME")

df_dim_pay.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_payment_type")

print("dim_payment_type created")

# ============================================================
# DIM_TIME_BLOCK
# ============================================================
df_tb = spark.table(f"{catalog}.{silver_schema}.trips") \
    .select("TIME_BLOCK").distinct().filter(col("TIME_BLOCK").isNotNull())

window_tb = Window.orderBy("TIME_BLOCK")

df_dim_tb = df_tb \
    .withColumn("TIME_BLOCK_SK", row_number().over(window_tb))

df_dim_tb.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_time_block")

print("dim_time_block created")

# ============================================================
# FACT_TRIPS
# ============================================================
df_t = spark.table(f"{catalog}.{silver_schema}.trips")

df_dv = spark.table("dim_vendor")
df_dz = spark.table("dim_taxi_zone")
df_dr = spark.table("dim_rate_code")
df_dp = spark.table("dim_payment_type")
df_tb = spark.table("dim_time_block")
df_dd = spark.table("dim_date")

df_fact = df_t.alias("t") \
    .join(df_dv.alias("v"),
        (col("t.VENDOR_ID") == col("v.VENDOR_ID")) &
        (col("t.TRIP_DATE") >= col("v.EFFECTIVE_DATE")) &
        (col("t.TRIP_DATE") <= col("v.END_DATE")),
        "left") \
    .join(df_dz.alias("pu"), col("t.PICKUP_LOCATION_ID") == col("pu.LOCATION_ID"), "left") \
    .join(df_dz.alias("do"), col("t.DROPOFF_LOCATION_ID") == col("do.LOCATION_ID"), "left") \
    .join(df_dr.alias("r"), col("t.RATE_CODE_ID") == col("r.RATE_CODE_ID"), "left") \
    .join(df_dp.alias("p"), col("t.PAYMENT_TYPE") == col("p.PAYMENT_TYPE_ID"), "left") \
    .join(df_tb.alias("tb"), col("t.TIME_BLOCK") == col("tb.TIME_BLOCK"), "left") \
    .join(df_dd.alias("d"), col("t.TRIP_DATE") == col("d.DATE_ACTUAL"), "left")

df_fact_trips = df_fact.select(
    monotonically_increasing_id().alias("TRIP_FACT_SK"),
    col("d.DATE_KEY"),
    col("v.VENDOR_SK"),
    col("pu.ZONE_SK").alias("PICKUP_ZONE_SK"),
    col("do.ZONE_SK").alias("DROPOFF_ZONE_SK"),
    col("r.RATE_CODE_SK"),
    col("p.PAYMENT_TYPE_SK"),
    col("tb.TIME_BLOCK_SK"),
    col("t.TRIP_DATE"),
    col("t.PASSENGER_COUNT"),
    col("t.TRIP_DISTANCE"),
    col("t.FARE_AMOUNT"),
    col("t.EXTRA"),
    col("t.TIP_AMOUNT"),
    col("t.TOLLS_AMOUNT"),
    col("t.TOTAL_AMOUNT")
)

df_fact_trips.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_trips")

print("fact_trips created")

# ============================================================
# FACT_DAILY_ZONE_SUMMARY
# ============================================================
df_summary = df_t.groupBy("TRIP_DATE", "PICKUP_LOCATION_ID") \
    .agg(
        count("*").alias("TOTAL_TRIPS"),
        sum("TOTAL_AMOUNT").alias("TOTAL_REVENUE")
    )

df_summary.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_daily_zone_summary")

print("fact_daily_zone_summary created")

# ============================================================
# VERIFY
# ============================================================
print("\n=== GOLD TABLES ===")
for t in ["dim_date","dim_vendor","dim_taxi_zone","dim_rate_code",
          "dim_payment_type","dim_time_block","fact_trips","fact_daily_zone_summary"]:
    print(f"{t}: {spark.table(t).count()} rows")